# 02 — Data Cleaning & ETL Pipeline
**Every transformation is logged below.**

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings, logging
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger()

df = pd.read_excel("../data/raw/Job Datsset.xlsx", engine="openpyxl")
log.info(f"[LOAD] {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

## 2. Standardise Column Names

In [ ]:
df.columns = (df.columns.str.strip().str.lower()
               .str.replace(' ', '_').str.replace(r'[^a-z0-9_]','',regex=True))
log.info(f"[RENAME] Columns → {list(df.columns)}")

## 3. Drop Duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates()
log.info(f"[DEDUP] Removed {before - len(df):,} duplicates. Remaining: {len(df):,}")

## 4. Identify & Rename Columns

In [ ]:
col_map = {}
for c in df.columns:
    if 'user' in c and 'id' in c: col_map[c]='user_id'
    elif 'job' in c and 'id' in c: col_map[c]='job_id'
    elif 'user' in c and 'skill' in c: col_map[c]='user_skills'
    elif 'req' in c or ('job' in c and 'skill' in c): col_map[c]='job_requirements'
    elif 'match' in c and 'score' in c: col_map[c]='match_score'
    elif 'recommend' in c: col_map[c]='recommended'
df = df.rename(columns=col_map)
log.info(f"[MAP] Final columns: {list(df.columns)}")

## 5. Handle Missing Values

In [ ]:
log.info(f"[NULL] Before drop:\n{df.isnull().sum()}")
before = len(df)
df = df.dropna(subset=['match_score','recommended'])
log.info(f"[NULL] Dropped {before-len(df):,} rows with null match_score/recommended")

## 6. Clean Skill Strings

In [ ]:
def clean_skills(s):
    if pd.isna(s): return ''
    return '|'.join(sk.strip().lower() for sk in str(s).split(',') if sk.strip())

df['user_skills']     = df['user_skills'].apply(clean_skills)
df['job_requirements']= df['job_requirements'].apply(clean_skills)
log.info("[SKILLS] Stripped whitespace, lowercased, re-joined with |")
df[['user_skills','job_requirements']].head(3)

## 7. Enforce Data Types

In [ ]:
df['match_score']  = pd.to_numeric(df['match_score'], errors='coerce')
df['recommended']  = pd.to_numeric(df['recommended'], errors='coerce').astype('Int64')
df = df.dropna(subset=['match_score','recommended'])
log.info(f"[DTYPE] match_score → float, recommended → Int64. Rows: {len(df):,}")

## 8. Feature Engineering

In [ ]:
df['user_skill_count']    = df['user_skills'].apply(lambda x: len(x.split(',')) if x else 0)
df['job_req_count']       = df['job_requirements'].apply(lambda x: len(x.split(',')) if x else 0)
df['skill_overlap_count'] = df.apply(
    lambda r: len(set(r['user_skills'].split(',')) & set(r['job_requirements'].split(','))) if r['user_skills'] and r['job_requirements'] else 0, axis=1)
df['match_score_band']    = pd.cut(df['match_score'], bins=[0,33,66,100], labels=['Low','Medium','High'], include_lowest=True)
df['skill_gap']           = df['job_req_count'] - df['skill_overlap_count']
df['overlap_ratio']       = (df['skill_overlap_count'] / df['job_req_count'].replace(0,1)).round(4)
log.info(f"[FEAT] Added 6 engineered columns. Final shape: {df.shape}")
df.head(3)

## 9. Export Cleaned Data

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/jobs_cleaned.csv', index=False)
log.info(f"[SAVE] Exported jobs_cleaned.csv — {len(df):,} rows, {df.shape[1]} columns")
print("✅ Cleaning complete. Output → data/processed/jobs_cleaned.csv")

## 10. Cleaning Summary
| Step | Action | Result |
|------|--------|--------|
| Dedup | Drop duplicates | Logged |
| Nulls | Drop null match_score / recommended | Logged |
| Skills | Lowercase + strip + rejoin | Done |
| Types | Enforce float / Int64 | Done |
| Features | +6 engineered columns | Done |